In [41]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import statsmodels.api as sm


In [58]:
file_path = r"C:\Users\rajeshkumar.t\Desktop\ML\tranctions.csv"
df=pd.read_csv(file_path, low_memory=False)
df = df.drop(columns=["tot_eff_amt_paymnt"])

In [59]:
print(df.columns)

Index(['transaction_yr', 'transaction_mth', 'transaction_wk', 'transaction_dt',
       'transaction_status', 'transaction_source', 'live_response_code',
       'payment_instrument', 'bank_code', 'hyp_flag', 'asp_flag',
       'merchant_id', 'merchant_status', 'marketplace_id', 'pg_id',
       'flipkart_emi_flag', 'marketplace_context', 'is_shopsy_order',
       'emi_flag', 'adonc_flag', 'count_of_tx', 'acct_cnt', 'tot_amt'],
      dtype='object')


In [60]:
for col in df.columns:
    if df[col].dtypes=='object':
        df[col]=df[col].astype('category').cat.codes
    


In [61]:
target_col = 'count_of_tx'
threshold = 0.05
correlations = df.corrwith(df[target_col])
selected_features = correlations[abs(correlations)> threshold].index.tolist()

if target_col in selected_features:
    selected_features.remove(target_col)
    

In [62]:
print(f"Original columns: {len(df.columns)}")
print(f"Selected columns: {len(selected_features)}")
print(f"Kept columns: {selected_features}")
print(f"Dropped feature: {list(set(df.columns) - set(selected_features) - set([target_col]))}")

Original columns: 23
Selected columns: 2
Kept columns: ['acct_cnt', 'tot_amt']
Dropped feature: ['emi_flag', 'transaction_status', 'transaction_source', 'pg_id', 'transaction_yr', 'payment_instrument', 'hyp_flag', 'marketplace_id', 'transaction_wk', 'is_shopsy_order', 'marketplace_context', 'transaction_dt', 'live_response_code', 'transaction_mth', 'merchant_id', 'flipkart_emi_flag', 'bank_code', 'asp_flag', 'merchant_status', 'adonc_flag']


In [63]:
X = df[selected_features]
Y = df[target_col]
X = sm.add_constant(X)
model = sm.OLS(Y, X).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:            count_of_tx   R-squared:                       0.912
Model:                            OLS   Adj. R-squared:                  0.912
Method:                 Least Squares   F-statistic:                 9.618e+06
Date:                Sat, 24 Jan 2026   Prob (F-statistic):               0.00
Time:                        23:34:08   Log-Likelihood:            -1.7471e+07
No. Observations:             1858498   AIC:                         3.494e+07
Df Residuals:                 1858495   BIC:                         3.494e+07
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          7.0164      2.149      3.265      0.0